# Cobrabox Example: Reciprocal Connectivity – #1 Data preparation

Authors: *[COBRA group](https://cobra.cs.cas.cz), Institute of Computer Science, The Czech Academy of Sciences*

<div align="left">
<img src="Images/Logo_CAS_ICS.png" align="left" width="254" alt="logo ICS">
</div>


<br>
<br>

---------------------

> TODO: Fix from here

This notebook runs the full batch pipeline for all 20 subjects:
1. Extract notch-filtered bipolar segments and save to disk
2. Compute band-averaged connectivity matrices (PDC by default) and save
3. Compute Reciprocal Connectivity (RC) per band and save

Each step checks whether output already exists and skips if so, making it safe to re-run
after interruptions.

## Import dependencies
> TODO: state the dependencies and point to documentation for installation.

This Notebook requires a ***python*** (>=3.11) installation together with ***xarray*** (>2026.2.0) and ***cobrabox*** (>=X.Y). For visualization it employs ***Matplotlib***. Please make sure these packages are installed in the python environment this notebook is running.


In [1]:
# Standard library imports
from timeit import default_timer as timer
from pathlib import Path

# Third-partiy imports
import cobrabox as cb

# Local libraries and modules
from utils import *


Describe why next cell

In [3]:
# Some useful comment here
cb.set_dataset_dir(Path(".") / "data", persist=False)

# Some useful comment here
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 21)]
CONNECTIVITY_METHOD = "pdc"  # change here to try other methods

## Step 1: Segment Extraction

For each subject we randomly sample `N_SEGMENTS = 20` non-overlapping 3-second windows
from all available runs. Each segment is:
- Notch-filtered at 50 Hz (on unipolar signals)
- Converted to bipolar montage

Segments are stored as a 3-D NetCDF array `(segment × space × time)` in `data/segments/`.
Already-saved subjects are skipped.

In [ ]:
patient_info = load_patient_info()

for subject_id in SUBJECTS:
    out_path = SEGMENTS_DIR / f"{subject_id}_segments.nc"
    if out_path.exists():
        print(f"{subject_id}: already saved, skipping")
        continue

    print(f"{subject_id}: loading data...", end=" ", flush=True)
    ds = cb.load_dataset("zurich_ieeg", subset=[subject_id])

    print("extracting segments...", end=" ", flush=True)
    rng = np.random.default_rng(42)
    segments = extract_segments(ds, subject_id, patient_info, rng=rng)

    save_segments(segments, subject_id)
    print(f"saved {len(segments)} segments → {out_path}")

    del ds, segments

## Step 2: Connectivity Computation

For each subject we load the saved segments and compute **Partial Directed Coherence (PDC)**
using a Vector Autoregressive (VAR) model of order `VAR_ORDER = 10`. PDC quantifies the
directed influence from channel j to channel i at each frequency.

The connectivity matrices are averaged:
1. Over the 20 segments (reducing noise)
2. Over each frequency band (delta, theta, alpha, beta, low_gamma, high_gamma, ripples, fast_ripples)

The result per subject is a dict of `(space_to × space_from)` matrices, one per band,
stored in `data/connectivity/`. The method name is embedded in the filename so you can
store results for multiple methods side by side.

In [ ]:
for subject_id in SUBJECTS:
    out_path = CONNECTIVITY_DIR / f"{subject_id}_{CONNECTIVITY_METHOD}_connectivity.nc"
    if out_path.exists():
        print(f"{subject_id}: already saved, skipping")
        continue

    t0 = timeit()
    print(f"{subject_id}: loading segments...", end=" ", flush=True)
    segments = load_segments(subject_id)

    print(f"computing {CONNECTIVITY_METHOD.upper()}...", end=" ", flush=True)
    avg_conn = compute_band_connectivity(segments, method=CONNECTIVITY_METHOD)

    save_connectivity(avg_conn, subject_id, method=CONNECTIVITY_METHOD)
    t1 = timeit()
    print(f"saved → {out_path}  ({t1-t0:.3f} s)")

    del segments, avg_conn

## Step 3: Reciprocal Connectivity

**Reciprocal Connectivity (RC)** summarises the directed connectivity matrix into a single
scalar per channel:

$$\text{RC}[i] = \text{in-strength}[i] - \text{out-strength}[i]$$

A **positive RC** means the channel receives more influence than it sends → net **sink**
(epileptic driver hypothesis: resected channels should be net sinks).

RC is computed per band from the pre-averaged connectivity matrices and saved to `data/rc/`.

In [ ]:
n_saved = 0
for subject_id in SUBJECTS:
    out_path = RC_DIR / f"{subject_id}_{CONNECTIVITY_METHOD}_rc.nc"
    if out_path.exists():
        print(f"{subject_id}: already saved, skipping")
        n_saved += 1
        continue

    print(f"{subject_id}: computing RC...", end=" ", flush=True)
    avg_conn = load_connectivity(subject_id, method=CONNECTIVITY_METHOD)
    rc_values = compute_rc(avg_conn)
    save_rc(rc_values, subject_id, method=CONNECTIVITY_METHOD)
    n_saved += 1
    print(f"saved → {out_path}")

print(f"\nDone. RC saved for {n_saved} subjects.")

## Verification

Spot-check the output for sub-01: compare the mean RC of resected vs non-resected channels
in each frequency band. If the hypothesis holds, the resected column should be consistently
higher (positive direction).

In [ ]:
subject_id = "sub-01"
rc_by_band = load_rc(subject_id, method=CONNECTIVITY_METHOD)
resected_pairs = set(patient_info[subject_id]["resected"])

print(f"{'band':<16} {'resected':>12} {'non-resected':>14} {'diff':>10}")
print("-" * 55)

for band_name, rc_da in rc_by_band.items():
    channels = rc_da.coords["space"].values
    resected_mask = np.array([ch in resected_pairs for ch in channels])

    if resected_mask.sum() == 0 or (~resected_mask).sum() == 0:
        print(f"{band_name:<16} {'N/A':>12} {'N/A':>14} {'N/A':>10}")
        continue

    mean_res = float(rc_da.values[resected_mask].mean())
    mean_non = float(rc_da.values[~resected_mask].mean())
    diff = mean_res - mean_non
    print(f"{band_name:<16} {mean_res:>12.4f} {mean_non:>14.4f} {diff:>10.4f}")